In [ ]:
!python -m pip install pyyaml==5.1
import sys, os, distutils.core
!git clone 'https://github.com/facebookresearch/detectron2'
dist = distutils.core.run_setup("./detectron2/setup.py")
!python -m pip install {' '.join([f"'{x}'" for x in dist.install_requires])}
sys.path.insert(0, os.path.abspath('./detectron2'))

In [ ]:
import torch, detectron2
!nvcc --version
TORCH_VERSION = ".".join(torch.__version__.split(".")[:2])
CUDA_VERSION = torch.__version__.split("+")[-1]
print("torch: ", TORCH_VERSION, "; cuda: ", CUDA_VERSION)
print("detectron2:", detectron2.__version__)

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2023 NVIDIA Corporation
Built on Tue_Aug_15_22:02:13_PDT_2023
Cuda compilation tools, release 12.2, V12.2.140
Build cuda_12.2.r12.2/compiler.33191640_0
torch:  2.1 ; cuda:  cu121
detectron2: 0.6


In [ ]:
# Some basic setup:
# Setup detectron2 logger
import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()

# import some common libraries
import numpy as np
import os, json, cv2, random
from google.colab.patches import cv2_imshow

# import some common detectron2 utilities
from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog, DatasetCatalog

In [ ]:
cfg = get_cfg()
# add project-specific config (e.g., TensorMask) here if you're not running a model in detectron2's core library
cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"))
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5  # set threshold for this model
# Find a model from detectron2's model zoo. You can use the https://dl.fbaipublicfiles... url as well
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")
predictor = DefaultPredictor(cfg)
outputs = predictor(im)

In [ ]:
v = Visualizer(im[:, :, ::-1], MetadataCatalog.get(cfg.DATASETS.TRAIN[0]), scale=1.2)
out = v.draw_instance_predictions(outputs["instances"].to("cpu"))

In [ ]:
# Obter os metadados do modelo
metadata = MetadataCatalog.get(cfg.DATASETS.TRAIN[0])

# Encontrar os índices das classes desejadas
indice_carro = metadata.thing_classes.index("car")
indice_moto = metadata.thing_classes.index("motorcycle")
indice_caminhao = metadata.thing_classes.index("truck")

print("Índice do carro:", indice_carro)
print("Índice da moto:", indice_moto)
print("Índice do caminhão:", indice_caminhao)

Índice do carro: 2
Índice da moto: 3
Índice do caminhão: 7


In [ ]:
instances = outputs["instances"]

car_instances = instances[instances.pred_classes == 2]
motocycle_instances = instances[instances.pred_classes == 3]
truck_instances = instances[instances.pred_classes == 7]

In [ ]:
num_car = len(car_instances)
num_motocycle = len(motocycle_instances)
num_truck = len(truck_instances)

In [ ]:
cfg = get_cfg()
# add project-specific config (e.g., TensorMask) here if you're not running a model in detectron2's core library
cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"))
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5  # set threshold for this model
# Find a model from detectron2's model zoo. You can use the https://dl.fbaipublicfiles... url as well
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")
predictor = DefaultPredictor(cfg)
outputs = predictor(im)

v = Visualizer(im[:, :, ::-1], MetadataCatalog.get(cfg.DATASETS.TRAIN[0]), scale=1.2)
out = v.draw_instance_predictions(outputs["instances"].to("cpu"))

# Obter os metadados do modelo
metadata = MetadataCatalog.get(cfg.DATASETS.TRAIN[0])

# Encontrar os índices das classes desejadas
indice_carro = metadata.thing_classes.index("car")
indice_moto = metadata.thing_classes.index("motorcycle")
indice_caminhao = metadata.thing_classes.index("truck")

print("Índice do carro:", indice_carro)
print("Índice da moto:", indice_moto)
print("Índice do caminhão:", indice_caminhao)

instances = outputs["instances"]

car_instances = instances[instances.pred_classes == 2]
motocycle_instances = instances[instances.pred_classes == 3]
truck_instances = instances[instances.pred_classes == 7]

num_car = len(car_instances)
num_motocycle = len(motocycle_instances)
num_truck = len(truck_instances)

font = cv2.FONT_HERSHEY_SIMPLEX
font_scale = 0.9
font_thickness = 2
font_color = (255, 0, 255)

# Texto a ser adicionado
text_car = f"Quantidade de carros: {num_car}"
text_moto = f"Quantidade de motos: {num_motocycle}"
text_truck = f"Quantidade de caminhoes: {num_truck}"

# Obtém a imagem com as previsões
output_image = out.get_image()[:, :, ::-1].copy()

x = 750

# Adicionar o texto na imagem copiada
cv2.putText(output_image, text_car, (x, 60), font, font_scale, font_color, font_thickness)
cv2.putText(output_image, text_moto, (x, 90), font, font_scale, font_color, font_thickness)
cv2.putText(output_image, text_truck, (x, 120), font, font_scale, font_color, font_thickness)

cv2_imshow(output_image)

In [ ]:
font = cv2.FONT_HERSHEY_SIMPLEX
font_scale = 0.9
font_thickness = 2
font_color = (255, 0, 255)

# Texto a ser adicionado
text_car = f"Quantidade de carros: {num_car}"
text_moto = f"Quantidade de motos: {num_motocycle}"
text_truck = f"Quantidade de caminhoes: {num_truck}"

# Obtém a imagem com as previsões
output_image = out.get_image()[:, :, ::-1].copy()

x = 750

# Adicionar o texto na imagem copiada
cv2.putText(output_image, text_car, (x, 60), font, font_scale, font_color, font_thickness)
cv2.putText(output_image, text_moto, (x, 90), font, font_scale, font_color, font_thickness)
cv2.putText(output_image, text_truck, (x, 120), font, font_scale, font_color, font_thickness)

cv2_imshow(output_image)

In [ ]:
#######################################################
                      #TEste
#######################################################
# Definindo as dimensões da imagem
height, width = output_image.shape[:2]

# Criando uma imagem de fundo preta
background = np.zeros((height, width, 3), dtype=np.uint8)

# Adicionando texto à imagem de fundo com fundo preto
cv2.putText(background, text_car, (x, 60), font, font_scale, (0, 0, 0), font_thickness)
cv2.putText(background, text_moto, (x, 90), font, font_scale, (0, 0, 0), font_thickness)
cv2.putText(background, text_truck, (x, 120), font, font_scale, (0, 0, 0), font_thickness)

# Criando uma máscara para o texto
mask = cv2.cvtColor(background, cv2.COLOR_BGR2GRAY)
_, mask = cv2.threshold(mask, 1, 255, cv2.THRESH_BINARY)

# Invertendo a máscara
mask_inv = cv2.bitwise_not(mask)

# Aplicando a máscara na imagem original
output_with_black_text_background = cv2.bitwise_and(output_image, output_image, mask=mask_inv)

# Adicionando o texto à imagem original
output_with_black_text_background = cv2.add(output_with_black_text_background, background)

# Mostrando a imagem resultante
cv2_imshow(output_with_black_text_background)